# Exploratory Data Analysis (EDA) — Complete Cheatsheet (All-in-One)

Merges everything from the Titanic + Heart Disease EDA notebooks into one reference.

**Sections**
1. Dataset Understanding
2. Feature Screening & Cleaning
3. Statistical Profiling
4. Univariate Analysis — Categorical
5. Univariate Analysis — Numerical
6. Outlier & Skewness Analysis
7. Multivariate Analysis (Cat-Cat, Cat-Num, Num-Num)
8. Correlation Heatmap + Pairplot
9. Automated EDA — `ydata_profiling`

**Datasets used**
- `titanic_data_updated.csv`  (in `Titanic/`)
- `heart.csv` (in `Heart Disease/`)
- `tips` (loaded from seaborn for scatterplot demo)

## 0. Setup — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

---
# 1. Dataset Understanding

We'll use **two datasets in parallel** so you can compare patterns:
- **Titanic** (classification: did the passenger survive?)
- **Heart Disease** (classification: does the patient have heart disease?)

In [ ]:
df_titanic = pd.read_csv('Titanic/titanic_data_updated.csv')
df_heart   = pd.read_csv('Heart Disease/heart.csv')

print('Titanic shape:', df_titanic.shape)
print('Heart shape  :', df_heart.shape)

In [ ]:
# Pick one as the working dataframe for most of the cheatsheet
df = df_titanic.copy()

print(f'Total potential features : {df.shape[1]}')
print(f'Total sample data        : {df.shape[0]}')

In [ ]:
# Two ways to peek at the data
df.head(10)

In [ ]:
# Random sample — more honest than .head() because it shows variety
df.sample(10)

### Heart Disease — column reference

| Column | Meaning | Type | Notes |
|---|---|---|---|
| `age` | Age | Numerical | years |
| `sex` | Gender | Categorical | `1=Male, 0=Female` |
| `cp` | Chest pain type | Categorical | |
| `trestbps` | Resting BP | Numerical | mm Hg |
| `chol` | Cholesterol | Numerical | mg/dl |
| `fbs` | Fasting blood sugar | Binary | `1=>120 mg/dl` |
| `restecg` | Resting ECG | Categorical | |
| `thalach` | Max heart rate | Numerical | |
| `exang` | Exercise-induced angina | Binary | |
| `oldpeak` | ST depression | Numerical | |
| `slope` | Slope of ST segment | Categorical | |
| `ca` | Major vessels (0–3) | Numerical | |
| `thal` | Thalassemia | Categorical | |
| `target` | Heart disease present | Target | `1=Yes, 0=No` |

---
# 2. Feature Screening & Cleaning

### 2.1 Missing values — count + heatmap

In [ ]:
# Column-wise count of NaNs
df.isnull().sum()

In [ ]:
# Visual heatmap — yellow stripes = missing
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing-value heatmap')
plt.show()

### 2.2 Duplicate rows — count + drop

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Shape after dedup:', df.shape)

### 2.3 Auto-categorize columns by dtype

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(exclude=['object']).columns.tolist()

print('Categorical:', cat_cols)
print('Numerical  :', num_cols)

---
# 3. Statistical Profiling

Two one-liners that tell you a lot about the data.

In [ ]:
# dtypes + non-null counts + memory usage
df.info()

In [ ]:
# count, mean, std, min, quartiles, max — for numerical columns
df.describe()

In [ ]:
# Same for categorical columns
df.describe(include='object')

---
# 4. Univariate Analysis — Categorical Columns

**Pattern:** `countplot` + percentage breakdown + optional pie chart.

### 4.1 Target column (`Survived`)

In [ ]:
sns.countplot(data=df, x='Survived')
plt.title('Target distribution — Survived')
plt.show()

print((df['Survived'].value_counts() / len(df)) * 100)

In [ ]:
# Pie chart with auto percentages
survived_count = df['Survived'].value_counts()
plt.pie(survived_count, labels=df['Survived'].unique(), autopct='%1.1f%%')
plt.title('Pie chart of survival rate')
plt.show()

### 4.2 Other categorical features — `Pclass`, `Sex`, `Embarked`

In [ ]:
for col in ['Pclass', 'Sex', 'Embarked']:
    fig, ax = plt.subplots(figsize=(5, 3))
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(f'Distribution of {col}')
    plt.show()
    print(f'--- {col} % ---')
    print((df[col].value_counts() / len(df)) * 100)
    print()

### 4.3 Same idea on Heart Disease — `target` + `sex`

In [ ]:
df_heart.drop_duplicates(inplace=True)

# Count + percentage in one table
target_ana = pd.DataFrame({
    'Count'     : df_heart['target'].value_counts(),
    'Percentage': df_heart['target'].value_counts(normalize=True),
})
target_ana

In [ ]:
# Pie with custom labels
plt.pie(df_heart['target'].value_counts(),
        labels=['No Disease', 'Disease Present'],
        autopct='%1.2f%%')
plt.title('Target Distribution — Heart Disease')
plt.show()

sns.countplot(data=df_heart, x='sex')
plt.title('Gender Analysis')
plt.show()

---
# 5. Univariate Analysis — Numerical Columns

**Pattern:** `histplot` (frequency) + `kdeplot` (probability density) + `boxplot` (spread + outliers).

### 5.1 `Age` — histogram + KDE

In [ ]:
sns.histplot(df['Age'], bins=50)
plt.title('Histogram of Age'); plt.xlabel('Ages'); plt.ylabel('Frequency')
plt.show()

sns.kdeplot(df['Age'])
plt.title('KDE plot of Age'); plt.xlabel('Ages'); plt.ylabel('Probability density')
plt.show()

### 5.2 `Fare` — histogram + KDE + boxplot (skewed column)

In [ ]:
sns.histplot(df['Fare'], bins=50); plt.title('Histogram of Fare'); plt.show()
sns.kdeplot(df['Fare']);             plt.title('KDE plot of Fare'); plt.show()
sns.boxplot(x=df['Fare']);           plt.title('Boxplot of Fare');   plt.show()

---
# 6. Outlier & Skewness Analysis

### 6.1 Boxplot — visual outlier check

In [ ]:
sns.boxplot(x=df_heart['chol'])
plt.title('Cholesterol Distribution')
plt.show()

### 6.2 IQR-based outlier detection
Lower bound = `Q1 - 1.5*IQR` | Upper bound = `Q3 + 1.5*IQR`

In [ ]:
q1 = df_heart['chol'].quantile(0.25)
q3 = df_heart['chol'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR bounds=[{lower}, {upper}]')
outliers = df_heart[(df_heart['chol'] < lower) | (df_heart['chol'] > upper)]
print(f'Outliers: {len(outliers)}')
outliers.head()

### 6.3 Skewness check
- **0** → symmetric 
- **> 0** → right-skewed (long tail on right) 
- **< 0** → left-skewed

In [ ]:
print('Age skew (Heart) :', df_heart['age'].skew())
print('Fare skew (Titanic):', df['Fare'].skew())

---
# 7. Multivariate Analysis

Used to find relationships between **two or more** variables.

| Combination | Plot |
|---|---|
| Cat-Cat | `countplot` with `hue` |
| Cat-Num | `barplot`, `boxplot`, `kdeplot` with `hue` |
| Num-Num | `scatterplot` |

### 7.1 Cat-Cat — `Sex` vs `Survived`, `Pclass` vs `Survived`

In [ ]:
sns.countplot(x=df['Sex'], hue=df['Survived'])
plt.title('Sex vs Survived'); plt.show()

# Group-wise proportion — tells you the survival RATE within each gender
print(df.groupby('Sex')['Survived'].value_counts(normalize=True))

In [ ]:
sns.countplot(x=df['Pclass'], hue=df['Survived'])
plt.title('Pclass vs Survived'); plt.show()

print(df.groupby('Pclass')['Survived'].value_counts(normalize=True))

### 7.2 Cat-Cat on Heart — `sex` vs `target`, `cp` vs `target`

In [ ]:
sns.countplot(data=df_heart, x='sex', hue='target')
plt.title('Gender vs Heart Disease'); plt.show()
print(df_heart.groupby('sex')['target'].value_counts(normalize=True))

In [ ]:
sns.countplot(data=df_heart, x='cp', hue='target')
plt.title('Chest Pain vs Heart Disease'); plt.show()

### 7.3 Cat-Num — `barplot` (compares means)

In [ ]:
sns.barplot(x=df['Pclass'], y=df['Fare'])
plt.title('Avg Fare per Pclass'); plt.show()

sns.barplot(x=df['Survived'], y=df['Age'], hue=df['Sex'])
plt.title('Avg Age by Survived, split by Sex'); plt.show()

sns.barplot(data=df_heart, x='target', y='chol')
plt.title('Avg Cholesterol by Heart Disease status'); plt.show()

### 7.4 Cat-Num — `kdeplot` with `hue` (compares distributions)
Shows the **shape** of a numerical column for each category — better than barplot for skewed data.

In [ ]:
sns.kdeplot(data=df, x='Age', hue='Survived')
plt.title('Age distribution by Survival'); plt.show()

### 7.5 Num-Num — `scatterplot`

In [ ]:
# Heart: Age vs Cholesterol, colored by target
sns.scatterplot(data=df_heart, x='age', y='chol', hue='target')
plt.title('Age vs Cholesterol — colored by heart disease'); plt.show()

In [ ]:
# Classic scatter demo on seaborn's tips dataset
tips = sns.load_dataset('tips')
sns.scatterplot(data=tips, x='total_bill', y='tip', hue='sex', size='size')
plt.title('Tips — bill vs tip, hue=sex, size=party size')
plt.show()

---
# 8. Correlation Heatmap + Pairplot

Quickly find **which features are related** to each other and to the target.

In [ ]:
# Pairplot — scatter matrix for a small set of numeric columns + target
sns.pairplot(df_heart[['age', 'chol', 'thalach', 'target']], hue='target')
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
plt.figure(figsize=(10, 10))
sns.heatmap(df_heart.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation heatmap — Heart Disease')
plt.show()

---
# 9. Automated EDA — `ydata_profiling`

Generates a full HTML report (missing-value matrix, distributions, correlations, interactions, warnings) in one call.

Install once: `pip install ydata-profiling`

In [ ]:
# Uncomment after installing:
# from ydata_profiling import ProfileReport
# profile = ProfileReport(df_titanic, title='Titanic Profile Report', explorative=True)
# profile.to_file('Titanic/report.html')
# profile  # displays inline in Jupyter
print('Pre-generated reports already exist:')
print(' - Titanic/report.html')
print(' - Heart Disease/report.html')

---
## Quick-reference summary

**Standard EDA workflow** for any dataset:

1. **Load & shape** — `read_csv`, `df.shape`, `df.head()`, `df.sample()`
2. **Screen** — `isnull().sum()` / heatmap, `duplicated().sum()` → `drop_duplicates`
3. **Categorize** — `select_dtypes` into `cat_cols`, `num_cols`
4. **Profile** — `df.info()`, `df.describe()`, `df.describe(include='object')`
5. **Univariate**
   - Categorical → `countplot` + `value_counts(normalize=True)` + pie
   - Numerical → `histplot` + `kdeplot` + `boxplot`
6. **Outlier / skew** — IQR bounds + `.skew()`
7. **Multivariate**
   - Cat-Cat → `countplot(hue=...)` + `groupby().value_counts(normalize=True)`
   - Cat-Num → `barplot`, `kdeplot(hue=...)`
   - Num-Num → `scatterplot`
8. **Big-picture** — `pairplot`, `heatmap(df.corr())`
9. **Automate** — `ydata_profiling.ProfileReport` for the full HTML

Done. Use this notebook as your single source of truth for EDA.